In [28]:
import httpx
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.expected_conditions import staleness_of
from selenium.webdriver.support.wait import WebDriverWait
from selenium.common.exceptions import TimeoutException
import pandas as pd
import numpy as np
import time

In [136]:
driver = webdriver.Firefox()
base = 'https://www.oddsportal.com'

In [137]:
driver.get(f'{base}/results/#esports')
wait = WebDriverWait(driver, timeout=5)
wait.until(EC.element_to_be_clickable((By.TAG_NAME, 'ul')))
soup = BeautifulSoup(driver.page_source)

In [138]:
wait = WebDriverWait(driver, timeout=10)

In [30]:
def wait_for_stable_count(driver, css_selector, timeout=10, interval=0.5):
    """Wait until element count stops changing."""
    import time
    deadline = time.time() + timeout
    prev_count = 0
    while time.time() < deadline:
        current_count = len(driver.find_elements(By.CSS_SELECTOR, css_selector))
        if current_count == prev_count and current_count > 0:
            return current_count
        prev_count = current_count
        time.sleep(interval)
    return prev_count

In [139]:
match_data = []
dota_tourneys = soup.find_all('ul', attrs={'data-testid': 'results-country-tournament-section'})[1]
original_window = driver.current_window_handle
for idx, a in enumerate(dota_tourneys.find_all('a')):
    href = a.get('href')
    if href == '/esports/dota-2/dota-2-dotapit-league-season-3/results/':
        print('skippnig first tourney')
        continue
    driver.get(f'{base}{href}')
    # year_selector = '.flex.flex-wrap.gap-2.py-3.text-xs.max-mm\\:flex-nowrap.max-mm\\:overflow-x-auto.max-mm\\:overflow-hidden.max-md\\:mx-3.max-sm\\:!hidden.no-scrollbar'
    # wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, year_selector)))
    tourney_soup = BeautifulSoup(driver.page_source, 'html.parser')
    years = tourney_soup.find(
            'div', 
            class_='flex flex-wrap gap-2 py-3 text-xs max-mm:flex-nowrap max-mm:overflow-x-auto max-mm:overflow-hidden max-md:mx-3 max-sm:!hidden no-scrollbar'
        ).find_all('a')
    for year in years:
        year_href = year.get('href')
        if year_href != driver.current_url:
            driver.get(year_href)
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
            count = wait_for_stable_count(driver, '.eventRow.flex.w-full.flex-col.text-xs', interval=2.5)
        except TimeoutException:
            print('skipping year')
            break
            continue
        time.sleep(1)
        try:
            pagination = driver.find_element(By.CSS_SELECTOR, '.pagination.my-7.flex.items-center.justify-center')
        except:
            pass
        while True:
            ##Code for getting values
            year_soup = BeautifulSoup(driver.page_source, 'html.parser')
            match_divs = year_soup.find_all('div', class_='eventRow flex w-full flex-col text-xs')
            for match_div in match_divs:
                odds_data = {}
                date = match_div.find('div', class_='flex w-full min-w-0 border-l border-r border-black-borders bg-gray-light')
                if date:
                    current_date = date.find('div').get_text()
                data = match_div.find('div', class_='border-black-borders border-b border-l border-r hover:bg-[#f9e9cc]').find_all('p')
                odds_data['date'] = current_date
                odds_data['time'] = data[0].get_text()
                odds_data['home_team'] = data[1].get_text()
                odds_data['away_team'] = data[2].get_text()
                home_odds = data[3].get_text()
                away_odds = data[4].get_text()
                if home_odds == '-' or away_odds == '-':
                    match_a = match_div.find('a', class_='next-m:flex next-m:!mt-0 ml-2 min-h-[32px] w-full hover:cursor-pointer')
                    match_href = match_a.get('href')
                    driver.switch_to.new_window('tab')
                    driver.get(f'{base}{match_href}')
                    count = wait_for_stable_count(driver, 'div.flex.flex-col[data-v-925bcd68]', interval=1.5)
                    match_soup = BeautifulSoup(driver.page_source, 'html.parser')
                    bookmaker_row = match_soup.find('div', attrs={'data-testid': 'over-under-expanded-row'})
                    odds = bookmaker_row.find_all('p', class_='odds-text line-through')
                    home_odds = odds[0].get_text()
                    away_odds = odds[1].get_text()
                    driver.close()
                    driver.switch_to.window(original_window)
                odds_data['home_odds'] = home_odds
                odds_data['away_odds'] = away_odds
                match_data.append(odds_data)
            try:
                driver.execute_script("arguments[0].scrollIntoView();", pagination)
                pagination.find_element(By.XPATH, '//a[text()="Next"]').click()
                count = wait_for_stable_count(driver, '.eventRow.flex.w-full.flex-col.text-xs', interval=1.5)
            except:
                print('pages finished')
                break
        break 
    break
    if idx == 1:
        break
print(len(match_data))
print(pd.DataFrame(match_data))

skippnig first tourney
pages finished
58
                           date   time        home_team        away_team  \
0      14 Sep 2025  - Play Offs  15:00          Falcons    Xtreme Gaming   
1      14 Sep 2025  - Play Offs  10:00       PARIVISION    Xtreme Gaming   
2      13 Sep 2025  - Play Offs  19:00     BetBoom Team    Xtreme Gaming   
3      13 Sep 2025  - Play Offs  16:00       PARIVISION          Falcons   
4      13 Sep 2025  - Play Offs  13:00     BetBoom Team           Heroic   
5      13 Sep 2025  - Play Offs  10:00    Xtreme Gaming     Nigma Galaxy   
6      12 Sep 2025  - Play Offs  19:00          Falcons     BetBoom Team   
7      12 Sep 2025  - Play Offs  15:20    Xtreme Gaming       PARIVISION   
8      12 Sep 2025  - Play Offs  13:00   Team Tidebound     Nigma Galaxy   
9      12 Sep 2025  - Play Offs  10:00   Tundra Esports           Heroic   
10     11 Sep 2025  - Play Offs  19:00     BetBoom Team     Nigma Galaxy   
11     11 Sep 2025  - Play Offs  16:45   Team T

In [99]:
match_div = BeautifulSoup('<div data-v-f69a8221="" id="QqRdwEzD" class="eventRow flex w-full flex-col text-xs" set="25235"><!----><div data-v-f69a8221="" class="flex w-full min-w-0 border-l border-r border-black-borders bg-gray-light" data-testid="secondary-header"><div class="flex w-full min-w-0 items-center justify-start border-black-borders pl-2" data-testid="date-header"><div class="w-full truncate font-main text-xs font-normal leading-5 text-black-main">22 Sep 2015  - Qualification</div></div><div class="flex h-5 min-sx:h-6"><div class="flex-center border-black-borders truncate border-l text-xs min-w-[60px]" data-testid="betting-tip-header">1</div><div class="flex-center border-black-borders truncate border-l text-xs min-w-[60px]" data-testid="betting-tip-header">2</div></div></div><div data-v-f69a8221="" class="border-black-borders border-b border-l border-r hover:bg-[#f9e9cc]"><div class="group flex" data-testid="game-row"><a class="next-m:flex next-m:!mt-0 ml-2 min-h-[32px] w-full hover:cursor-pointer" href="/esports/dota-2/dota-2-esl-one-new-york/vici-gaming-potential-dota-2-ehome-dota-2-QqRdwEzD/"><div class="column max-mt:gap-2 max-mt:py-2 flex h-full w-full items-center" data-testid="game-row"><div data-v-56ca3c46="" class="flex items-center max-sm:shrink-0 max-sx:flex-col max-sx:gap-2 basis-[10%] max-sm:basis-auto"><!----><div data-v-56ca3c46="" class="flex flex-row items-center text-[12px] text-gray-dark next-m:flex-col min-md:flex-row min-md:gap-1 w-full" data-testid="time-item"><div data-v-56ca3c46="" class="flex w-full"><p data-v-56ca3c46="">15:30</p><span data-v-56ca3c46="" class="ml-auto pr-2 max-sm:!hidden next-m:!hidden"></span><!----></div></div><!----></div><div class="flex w-full items-center"><div data-v-23626bce="" data-testid="event-participants" class="align-center mx-1 flex w-full flex-col items-center gap-1"><div data-v-23626bce="" class="flex w-full min-w-0 flex-col gap-1 pt-[2px] text-xs leading-[16px] max-mt:pl-1 min-mt:!flex-row min-mt:!gap-2 justify-center"><a data-v-23626bce="" class="flex min-w-0 basis-[50%] cursor-pointer items-start justify-start gap-1 overflow-hidden min-mt:!justify-end" title="Vici Gaming Potential"><img data-v-23626bce="" src="/serve/images/team-logo/Men/0vklummh-Sb1V1CzA.png?260320150504" alt="Vici Gaming Potential" class="min-mt:order-3 h-[18px] w-[18px]" loading="lazy"><div data-v-23626bce="" class="min-w-0 whitespace-nowrap group-hover:underline min-md:overflow-hidden"><p data-v-23626bce="" class="participant-name truncate">Vici Gaming Potential</p></div><div data-v-23626bce="" class="ml-auto mr-3 flex font-bold min-mt:!hidden">0</div></a><div data-v-23626bce="" class="relative flex text-gray-dark"><div data-v-23626bce="" class="flex gap-1 font-bold font-bold"><div data-v-23626bce="" class="hidden min-mt:!flex">0</div><a data-v-23626bce="" class="hidden cursor-pointer min-mt:!flex">–</a><div data-v-23626bce="" class="hidden min-mt:!flex font-bold">2</div></div><!----></div><a data-v-23626bce="" class="justify-content flex basis-[50%] cursor-pointer items-center gap-1 overflow-hidden min-mt:!gap-2" title="EHOME"><img data-v-23626bce="" src="/serve/images/team-logo/Men/nD8VaRk9-GnK49tIm.png?260320150504" alt="EHOME" class="h-[18px] w-[18px]" loading="lazy"><div data-v-23626bce="" class="min-w-0 whitespace-nowrap group-hover:underline min-md:truncate font-bold"><p data-v-23626bce="" class="participant-name truncate">EHOME</p></div><div data-v-23626bce="" class="ml-auto mr-3 flex font-bold min-mt:!hidden font-bold">2</div></a></div><!----><!----></div></div></div><div class="mr-1.5 flex basis-[10%] flex-wrap items-center justify-end" data-testid="game-status-box"><!----><!----><!----></div></a><div class="flex-center border-black-borders min-w-[60px] flex-col gap-1 border-l pb-1 pt-1"><span class="border-black-borders next-m:min-h-[26px] next-m:min-w-[80%] default-odds-bg-bgcolor flex min-h-[28px] min-w-[50px] items-center justify-center border font-bold"><p class="height-content">-</p></span></div><div class="flex-center border-black-borders min-w-[60px] flex-col gap-1 border-l pb-1 pt-1"><span class="border-black-borders next-m:min-h-[26px] next-m:min-w-[80%] default-odds-bg-bgcolor flex min-h-[28px] min-w-[50px] items-center justify-center border font-bold"><p class="height-content">-</p></span></div></div></div></div>', 'html.parser')

In [103]:
driver.switch_to.window(original_window)

In [ ]:
odds_data = {}
date = match_div.find('div', class_='flex w-full min-w-0 border-l border-r border-black-borders bg-gray-light')
if date:
    current_date = date.find('div').get_text()
data = match_div.find('div', class_='border-black-borders border-b border-l border-r hover:bg-[#f9e9cc]').find_all('p')
odds_data['date'] = current_date
odds_data['time'] = data[0].get_text()
odds_data['home_team'] = data[1].get_text()
odds_data['away_team'] = data[2].get_text()
home_odds = data[3].get_text()
away_odds = data[4].get_text()
if home_odds == '-' or away_odds == '-':
    match_a = match_div.find('a', class_='next-m:flex next-m:!mt-0 ml-2 min-h-[32px] w-full hover:cursor-pointer')
    match_href = match_a.get('href')
    driver.switch_to.new_window('tab')
    driver.get(f'{base}{match_href}')
    count = wait_for_stable_count(driver, 'div.flex.flex-col[data-v-925bcd68]', interval=1.5)
    match_soup = BeautifulSoup(driver.page_source, 'html.parser')
    bookmaker_row = match_soup.find('div', attrs={'data-testid': 'over-under-expanded-row'})
    odds = bookmaker_row.find_all('p', class_='odds-text line-through')
    home_odds = odds[0].get_text()
    away_odds = odds[1].get_text()
    driver.close()
    driver.switch_to.window(original_window)
odds_data['home_odds'] = home_odds
odds_data['away_odds'] = away_odds